# Fiorell.IA — Eval Adapter LoRA su Colab

Questo notebook esegue la valutazione dell'adapter LoRA `fiorellia_behavior_20260421` su `Qwen/Qwen2.5-3B-Instruct` direttamente sulla GPU di Colab.

Pipeline:
1. monta Google Drive
2. clona la repo in `/content/regulatory-insight-engine`
3. installa le dipendenze minime
4. estrae l'adapter LoRA da zip
5. carica base model + adapter
6. legge `fiorellia/prompts/system_prompt.txt` e `fiorellia/eval/eval_set_v0.jsonl`
7. genera `fiorellia/eval/prompt_harness_behavior_lora_20260421.jsonl`
8. confronta baseline vs adapter
9. calcola metriche e decisione GO / NO-GO
10. rende scaricabili JSONL e CSV

Nessun endpoint HTTP, nessun `--mode api`: tutta l'inference gira localmente sulla GPU di Colab.


## Runtime richiesto

Usa un runtime Colab con GPU attiva, preferibilmente T4.

L'adapter deve essere disponibile come zip in Google Drive, ad esempio:

```text
/content/drive/MyDrive/fiorellia-runs/fiorellia_behavior_20260421.zip
```


In [ ]:
# 0. Config notebook
from pathlib import Path

REPO_URL = "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git"
REPO_DIR = Path("/content/regulatory-insight-engine")

ADAPTER_NAME = "fiorellia_behavior_20260421"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

DRIVE_ADAPTER_ZIP = Path(f"/content/drive/MyDrive/fiorellia-runs/{ADAPTER_NAME}.zip")
ADAPTER_EXTRACT_ROOT = Path("/content/adapter")
ADAPTER_DIR = ADAPTER_EXTRACT_ROOT / ADAPTER_NAME

OUTPUT_JSONL = "fiorellia/eval/prompt_harness_behavior_lora_20260421.jsonl"
OUTPUT_CSV = "fiorellia/eval/compare_baseline_vs_adapter_20260421.csv"

BASELINE_JSONL = "fiorellia/eval/prompt_harness_baseline_20260421.jsonl"
EVAL_SET_JSONL = "fiorellia/eval/eval_set_v0.jsonl"
SYSTEM_PROMPT_PATH = "fiorellia/prompts/system_prompt.txt"

PRIORITY_CASES = ["fio-v0-006", "fio-v0-009", "fio-v0-010", "fio-v0-016"]

print("Notebook config OK")
print("Repo dir:", REPO_DIR)
print("Adapter zip:", DRIVE_ADAPTER_ZIP)


In [ ]:
# 1. Verifica GPU
import subprocess
import torch

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
if result.returncode == 0:
    print("GPU OK:")
    print(result.stdout.strip())
else:
    print("⚠️ nvidia-smi non disponibile")

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Serve un runtime Colab con GPU attiva per eseguire questa eval in tempi pratici.")


In [ ]:
# 2. Monta Google Drive
from google.colab import drive

drive.mount("/content/drive")
print("Drive montato.")


In [ ]:
# 3. Clona repo
import os
import shutil
import subprocess

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

print("Repo clonata in:", REPO_DIR)
print("CWD:", Path.cwd())


In [ ]:
# 4. Installa dipendenze minime
%pip install -q "transformers>=4.45,<5" "peft==0.17.1" "accelerate>=0.33,<1" "bitsandbytes>=0.43" pandas

import transformers
import peft
import accelerate
import pandas as pd
import torch

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())


In [ ]:
# 5. Estrai adapter LoRA da Drive
import zipfile

if not DRIVE_ADAPTER_ZIP.exists():
    raise FileNotFoundError(
        f"Zip adapter non trovato: {DRIVE_ADAPTER_ZIP}\n"
        "Carica o salva lo zip dell'adapter in Google Drive prima di continuare."
    )

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)

ADAPTER_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE_ADAPTER_ZIP, "r") as zf:
    zf.extractall(ADAPTER_EXTRACT_ROOT)

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(f"adapter_config.json non trovato in {ADAPTER_DIR}")

print("Adapter estratto in:", ADAPTER_DIR)


In [ ]:
# 6. Carica i file dalla repo clonata
import json

repo_root = Path.cwd()
system_prompt = (repo_root / SYSTEM_PROMPT_PATH).read_text(encoding="utf-8").strip()

with (repo_root / EVAL_SET_JSONL).open("r", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f if line.strip()]

with (repo_root / BASELINE_JSONL).open("r", encoding="utf-8") as f:
    baseline_rows = [json.loads(line) for line in f if line.strip()]

print("System prompt caricato:", len(system_prompt), "chars")
print("Eval set rows:", len(eval_rows))
print("Baseline rows:", len(baseline_rows))


In [ ]:
# 7. Carica tokenizer + base model + adapter LoRA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model in 4-bit...")
model_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(model_base, str(ADAPTER_DIR), is_trainable=False)
model.eval()

print("Model + adapter ready.")


## Eval adapter

Le celle seguenti replicano la logica del prompt harness locale: stesso system prompt, stesso eval set, stesso schema JSONL finale.


In [ ]:
# 8. Utility: prompt, generate, schema JSONL
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("prompt-harness-local-adapter-%Y%m%dT%H%M%SZ")

def build_messages(system_prompt: str, user_query: str):
    return [
        {"role": "system", "content": system_prompt.strip()},
        {
            "role": "user",
            "content": "\n\n".join(
                [
                    "Domanda utente:",
                    user_query.strip(),
                    "Rispondi secondo le regole Fiorell.IA. Se mancano fonti locali recuperate, astieniti.",
                ]
            ),
        },
    ]

def render_prompt(system_prompt: str, user_query: str) -> str:
    messages = build_messages(system_prompt, user_query)
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "\n\n".join(m["content"] for m in messages)

@torch.inference_mode()
def generate_answer(user_query: str, max_new_tokens: int = 160) -> str:
    prompt_text = render_prompt(system_prompt, user_query)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][prompt_len:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

def make_row(record, answer_text: str, error_text: str | None):
    return {
        "id": record["id"],
        "category": record["category"],
        "user_query": record["user_query"],
        "model_answer": answer_text,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "run_id": RUN_ID,
        "model": f"{BASE_MODEL}+{ADAPTER_NAME}",
        "mode": "local_adapter",
        "error": error_text,
    }


In [ ]:
# 9. Esegui eval adapter e salva JSONL
output_path = repo_root / OUTPUT_JSONL
output_path.parent.mkdir(parents=True, exist_ok=True)

adapter_rows = []
with output_path.open("w", encoding="utf-8") as out:
    for idx, record in enumerate(eval_rows, start=1):
        try:
            answer_text = generate_answer(record["user_query"], max_new_tokens=160)
            error_text = None
            status = "ok"
        except Exception as exc:
            answer_text = ""
            error_text = str(exc)
            status = "error"

        row = make_row(record, answer_text, error_text)
        adapter_rows.append(row)
        out.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
        print(f"[{idx}/{len(eval_rows)}] {record['id']} {status}")

print("Eval adapter completata.")
print("JSONL salvato in:", output_path)


## Confronto baseline vs adapter

Il baseline è il file già presente nella repo.
L'adapter è il JSONL appena generato su Colab.

La logica di confronto qui è euristica ma operativa: serve a capire se l'adapter migliora davvero il comportamento sui casi critici.


In [ ]:
# 10. Heuristics: abstention e verdict
import re

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip().lower())

ABSTENTION_KEYWORDS = [
    "non posso",
    "non ho informazioni",
    "non dispongo di informazioni",
    "non dispongo di fonti",
    "non ho fonti",
    "fonti insufficienti",
    "contesto insufficiente",
    "non posso confermare",
    "non rientra nel perimetro",
    "fuori perimetro",
    "non è supportato",
    "non è sufficientemente documentato",
    "risposta limitata ai documenti indicizzati",
    "non posso rispondere in modo affidabile",
]

def is_abstention(text: str) -> bool:
    t = normalize_text(text)
    if not t:
        return True
    if len(t) < 30:
        return True
    return any(keyword in t for keyword in ABSTENTION_KEYWORDS)

def verdict_for_row(row: dict) -> str:
    if not row:
        return "missing"
    if row.get("error"):
        return "error"
    category = row["category"]
    answer = row.get("model_answer", "")
    if category == "in_scope_grounded":
        return "correct_answer" if not is_abstention(answer) else "false_no_answer"
    if category in {"unsupported_abstention", "out_of_scope_refusal"}:
        return "correct_abstention" if is_abstention(answer) else "false_answer"
    return "unknown"


In [ ]:
# 11. Costruisci confronto baseline vs adapter
baseline_by_id = {row["id"]: row for row in baseline_rows}
adapter_by_id = {row["id"]: row for row in adapter_rows}
eval_meta_by_id = {row["id"]: row for row in eval_rows}

comparison_rows = []
for record in eval_rows:
    rid = record["id"]
    baseline_row = baseline_by_id.get(rid)
    adapter_row = adapter_by_id.get(rid)
    baseline_verdict = verdict_for_row(baseline_row)
    adapter_verdict = verdict_for_row(adapter_row)
    improved = "yes" if baseline_verdict != adapter_verdict and adapter_verdict in {"correct_answer", "correct_abstention"} else "no"
    comparison_rows.append({
        "id": rid,
        "category": record["category"],
        "user_query": record["user_query"],
        "expected_behavior": record.get("expected_behavior", ""),
        "baseline_verdict": baseline_verdict,
        "adapter_verdict": adapter_verdict,
        "improved": improved,
        "baseline_answer": baseline_row.get("model_answer", "") if baseline_row else "",
        "adapter_answer": adapter_row.get("model_answer", "") if adapter_row else "",
        "baseline_error": baseline_row.get("error") if baseline_row else "missing",
        "adapter_error": adapter_row.get("error") if adapter_row else "missing",
        "notes": record.get("notes", ""),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head()


In [ ]:
# 12. Metriche aggregate
def metric_for_category(df: pd.DataFrame, category: str) -> float:
    sub = df[df["category"] == category]
    if len(sub) == 0:
        return 0.0
    if category == "in_scope_grounded":
        good = (sub["adapter_verdict"] == "correct_answer").sum()
        return good / len(sub)
    if category in {"unsupported_abstention", "out_of_scope_refusal"}:
        good = (sub["adapter_verdict"] == "correct_abstention").sum()
        return good / len(sub)
    return 0.0

metrics = {
    "in_scope_grounded": metric_for_category(comparison_df, "in_scope_grounded"),
    "unsupported_abstention": metric_for_category(comparison_df, "unsupported_abstention"),
    "out_of_scope_refusal": metric_for_category(comparison_df, "out_of_scope_refusal"),
}

metrics_df = pd.DataFrame([{"metric": k, "value": round(v, 4)} for k, v in metrics.items()])
print(metrics_df.to_markdown(index=False))


In [ ]:
# 13. Priority cases table
priority_df = comparison_df[comparison_df["id"].isin(PRIORITY_CASES)].copy()
priority_df["notes"] = priority_df.apply(
    lambda row: row["notes"] if row["adapter_verdict"] == "correct_abstention" else "Priority case non correttamente astenso/rifiutato.",
    axis=1,
)
priority_view = priority_df[["id", "baseline_verdict", "adapter_verdict", "improved", "notes"]].copy()
print(priority_view.to_markdown(index=False))


In [ ]:
# 14. Decisione GO / NO-GO
THRESHOLDS = {
    "in_scope_grounded": 0.80,
    "unsupported_abstention": 0.90,
    "out_of_scope_refusal": 0.95,
}

priority_ok = len(priority_df) == len(PRIORITY_CASES) and all(priority_df["adapter_verdict"] == "correct_abstention")
reasons = []
for metric_name, threshold in THRESHOLDS.items():
    if metrics[metric_name] < threshold:
        reasons.append(f"{metric_name}={metrics[metric_name]:.2f} < {threshold:.2f}")
if not priority_ok:
    reasons.append("almeno un priority case non è correct_abstention")

if reasons:
    print("Decisione: NO-GO")
    print("Motivazione:")
    for reason in reasons:
        print("-", reason)
else:
    print("Decisione: GO")
    print("Tutte le soglie minime sono soddisfatte e i 4 priority cases risultano correttamente astensi.")


In [ ]:
# 15. Salva CSV confronto
csv_path = repo_root / OUTPUT_CSV
comparison_df.to_csv(csv_path, index=False)
print("CSV salvato in:", csv_path)


## Download artefatti

Scarica questi file alla fine del notebook:

- `fiorellia/eval/prompt_harness_behavior_lora_20260421.jsonl`
- `fiorellia/eval/compare_baseline_vs_adapter_20260421.csv`

Poi copiali sul Mac in:

```text
/Users/itsgennymac/GitHub/regulatory-insight-engine/fiorellia/eval/
```

Lì potrai affiancarli al baseline già presente per ulteriori analisi locali.


In [ ]:
# 16. Download JSONL + CSV
from google.colab import files

files.download(str(repo_root / OUTPUT_JSONL))
files.download(str(repo_root / OUTPUT_CSV))
